# WhatIf — Anthology Edition
**Author: Swayam Shakya**

One Decision. Infinite Possibilities.

Extract the complete archive before opening this notebook. Use Python 3.10 or newer. Run cells from top to bottom. The website is served by Python on localhost; HTML does not execute Python directly. No network connection is needed after dependencies are installed.

This edition includes 105 short scenarios, 315 endings and 2,135 verified routes. Eleven original instrumental scores are bundled locally.

The root **index.html** is the main interface. This notebook starts its Python backend. For a quicker launch without opening Jupyter, use `START_WINDOWS.bat` or `bash START_MAC_LINUX.sh`. Both methods use the same saved journeys.

## 1. Locate the project
Open this notebook from the extracted `WhatIf` folder. This cell checks every required file before starting anything.

In [ ]:
from pathlib import Path
import sys
if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer.")
candidates = [Path.cwd(), Path.cwd() / "WhatIf"]
PROJECT = next((p.resolve() for p in candidates if (p / "app.py").exists() and (p / "stories/scenarios.py").exists()), None)
if PROJECT is None:
    raise RuntimeError("Open this notebook with the complete extracted WhatIf folder as your working directory.")
required = ["app.py", "launcher.py", "templates/index.html", "static/js/app.js", "static/js/theme.js", "static/css/style.css", "engine/decision_engine.py", "engine/story_validator.py", "stories/scenarios.py", "stories/anthology.py", "static/js/audio.js", "static/audio/main.wav", "static/audio/scores.json"]
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise RuntimeError("Missing project files: " + ", ".join(missing))
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)
print("Python:", sys.version.split()[0])

## 2. Check the server dependency
If Flask is missing or outside the supported major version, this cell installs it into the current notebook kernel. The first installation needs internet access. If installation fails, open a terminal in the project directory and run `python -m pip install -r requirements.txt`, then restart the kernel.

In [ ]:
import importlib.metadata
import subprocess
try:
    flask_version = importlib.metadata.version("Flask")
except importlib.metadata.PackageNotFoundError:
    flask_version = "0"
if flask_version.split(".")[0] != "3":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "Flask>=3.0,<4.0"])
print("Flask", importlib.metadata.version("Flask"))

## 3. Validate stories and initialise storage
This exhaustively walks every supported route and creates the database if necessary. Existing progress is preserved. Do not edit story definitions while a server is running. Stop and restart it after code changes.

In [ ]:
from stories.scenarios import SCENARIOS
from engine.story_validator import validate
from app import create_app
for story in SCENARIOS.values():
    paths = validate(story)
    print(story["title"], "—", len(paths), "complete routes")
application = create_app()
print("Database ready in", PROJECT / "database")
print("Stories:", len(SCENARIOS), "— audio is bundled in static/audio")

## 4. Start the website
Keep the notebook kernel running while you play. Re-running this cell reuses the running server. If the port is occupied, choose another number (such as 5051), or use `0` to select a free port. Use the exact `127.0.0.1` address shown to keep the same browser profile.

In [ ]:
from launcher import start_server, stop_server
from IPython.display import display, HTML
PORT = 5050
server = start_server(PORT, application)
url = f"http://127.0.0.1:{server.port}/index.html"
display(HTML(f'<p><a href="{url}" target="_blank" rel="noopener">Open WhatIf ↗</a></p>'))

## 5. Play, branch and compare
Tap, click or press a key anywhere to unlock the opening music. Scenario music switches automatically as you play. The ♫ button and bottom volume slider remain available throughout the game.

1. Open **Explore**, enter a scenario and choose a decision.
2. Open **Timeline Lab**, then branch before an earlier decision.
3. Make a different choice and compare the two journeys.
4. Save a checkpoint from the player; restore it as a separate branch in **My Journeys**.

Your browser cookie identifies an anonymous profile. Keep the cookie, `database/whatif.sqlite3`, and `database/.session-key` to retain access after a restart. Use the same browser and hostname.

**Remote notebooks:** This project is designed for Jupyter on your own computer. If a hosted notebook cannot expose localhost, download and extract the project locally, install requirements, and run `python app.py`. No external tunnel is required.

## 6. Optional checks
Tests use temporary databases and leave your saved journeys alone. This cell can run while the website is open.

In [ ]:
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError("A test failed. Review the output before using this version.")

## 7. Stop the website
Run this cell when finished. The database remains on disk. To play again, rerun cell 4. Running the whole notebook including this final cell starts and then stops the server as a complete lifecycle check.

In [ ]:
stop_server()